In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("health_data.csv")
print("Original shape:", df.shape)
df.head()

Original shape: (255, 12)


,Patient_ID,City,Gender,Admission_Type,Insurance_Type,Age,BMI,Fasting_Blood_Sugar,Medications_Count,Distance_To_Hospital_Km,Admission_Date,Hospital_Stay_Days
0,P1066,Mumbai,Male,Referral,Private,56,19.3,127.0,8,9.7,2026-05-30,9
1,P1231,Chennai,Female,Referral,Government,49,25.8,133.0,2,11.0,2025-12-31,6
2,P1106,Bengaluru,Male,Emergency,Uninsured,50,25.4,111.0,3,4.5,2025-07-21,7
3,P1167,Bengaluru,Male,Emergency,Government,69,28.5,123.0,3,35.4,2025-05-02,10
4,P1040,Delhi,Male,Planned,Uninsured,66,24.5,78.0,5,6.6,2026-07-04,9


In [3]:
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print(df["City"].value_counts())
print(df[(df["Age"] < 0) | (df["Age"] > 110)][["Patient_ID", "Age"]])

Patient_ID                 0
City                       0
Gender                     0
Admission_Type             0
Insurance_Type             5
Age                        0
BMI                        5
Fasting_Blood_Sugar        5
Medications_Count          0
Distance_To_Hospital_Km    5
Admission_Date             0
Hospital_Stay_Days         0
dtype: int64
Duplicates: 5
City
Mumbai         64
Delhi          63
Bengaluru      51
Chennai        44
Kolkata        26
delhi           1
bengaluru       1
Bengaluru       1
 Bengaluru      1
DELHI           1
 kolkata        1
MUMBAI          1
Name: count, dtype: int64
    Patient_ID  Age
111      P1096  -28
209      P1136  214


In [4]:
clean = df.copy()

# 1. Remove duplicate rows
clean = clean.drop_duplicates()
print("After removing duplicates:", clean.shape)

# 2. Fix City names (spaces and upper/lower case problems)
clean["City"] = clean["City"].str.strip().str.title()
print(clean["City"].value_counts())

After removing duplicates: (250, 12)
City
Mumbai       63
Delhi        63
Bengaluru    54
Chennai      43
Kolkata      27
Name: count, dtype: int64


In [5]:
# 3. Invalid ages (negative or above 110) -> treat as missing
clean.loc[(clean["Age"] < 0) | (clean["Age"] > 110), "Age"] = np.nan

# 4. Fill missing numeric values with the median
for col in ["Age", "BMI", "Fasting_Blood_Sugar", "Distance_To_Hospital_Km"]:
    clean[col] = clean[col].fillna(clean[col].median())

# 5. Fill missing Insurance_Type with the most common value (mode)
clean["Insurance_Type"] = clean["Insurance_Type"].fillna(clean["Insurance_Type"].mode()[0])

# 6. Fix data types
clean["Admission_Date"] = pd.to_datetime(clean["Admission_Date"])
clean["Age"] = clean["Age"].astype(int)

print(clean.isnull().sum())

Patient_ID                 0
City                       0
Gender                     0
Admission_Type             0
Insurance_Type             0
Age                        0
BMI                        0
Fasting_Blood_Sugar        0
Medications_Count          0
Distance_To_Hospital_Km    0
Admission_Date             0
Hospital_Stay_Days         0
dtype: int64


In [6]:
clean["Age_Group"] = pd.cut(clean["Age"], bins=[0, 18, 40, 60, 120],
                            labels=["Child/Teen", "Young Adult", "Middle Aged", "Senior"])

clean["BMI_Category"] = pd.cut(clean["BMI"], bins=[0, 18.5, 25, 30, 100],
                               labels=["Underweight", "Normal", "Overweight", "Obese"])

clean["Admission_Year"] = clean["Admission_Date"].dt.year
clean["Admission_Month"] = clean["Admission_Date"].dt.month

clean.head()

,Patient_ID,City,Gender,Admission_Type,Insurance_Type,Age,BMI,Fasting_Blood_Sugar,Medications_Count,Distance_To_Hospital_Km,Admission_Date,Hospital_Stay_Days,Age_Group,BMI_Category,Admission_Year,Admission_Month
0,P1066,Mumbai,Male,Referral,Private,56,19.3,127.0,8,9.7,2026-05-30,9,Middle Aged,Normal,2026,5
1,P1231,Chennai,Female,Referral,Government,49,25.8,133.0,2,11.0,2025-12-31,6,Middle Aged,Overweight,2025,12
2,P1106,Bengaluru,Male,Emergency,Uninsured,50,25.4,111.0,3,4.5,2025-07-21,7,Middle Aged,Overweight,2025,7
3,P1167,Bengaluru,Male,Emergency,Government,69,28.5,123.0,3,35.4,2025-05-02,10,Senior,Overweight,2025,5
4,P1040,Delhi,Male,Planned,Uninsured,66,24.5,78.0,5,6.6,2026-07-04,9,Senior,Normal,2026,7


In [7]:
print(clean.describe())

              Age         BMI  Fasting_Blood_Sugar  Medications_Count  \
count  250.000000  250.000000           250.000000          250.00000   
mean    45.376000   26.461200           105.432000            3.10800   
min     18.000000   16.000000            70.000000            0.00000   
25%     33.000000   23.800000            89.000000            2.00000   
50%     45.000000   26.600000           105.000000            3.00000   
75%     56.000000   29.300000           121.000000            4.00000   
max     83.000000   36.800000           195.000000            8.00000   
std     16.209058    4.297853            23.232485            1.69346   

       Distance_To_Hospital_Km              Admission_Date  \
count                250.00000                         250   
mean                  11.70000  2025-12-13 15:15:50.400000   
min                    1.00000         2024-11-23 00:00:00   
25%                    3.42500         2025-06-04 00:00:00   
50%                    8.40000  

In [8]:
print(clean.groupby("Admission_Type")["Hospital_Stay_Days"].mean().round(2))
print()
print(clean.groupby("Insurance_Type")["Hospital_Stay_Days"].mean().round(2))
print()
print(clean["Age_Group"].value_counts())
print()
print(clean["BMI_Category"].value_counts())

Admission_Type
Emergency    7.27
Planned      6.17
Referral     7.15
Name: Hospital_Stay_Days, dtype: float64

Insurance_Type
Government    6.82
Private       6.86
Uninsured     6.91
Name: Hospital_Stay_Days, dtype: float64

Age_Group
Middle Aged    100
Young Adult     89
Senior          48
Child/Teen      13
Name: count, dtype: int64

BMI_Category
Overweight     106
Normal          83
Obese           51
Underweight     10
Name: count, dtype: int64


In [9]:
# Correlation between numeric columns
cols = ["Age", "BMI", "Fasting_Blood_Sugar", "Medications_Count",
        "Distance_To_Hospital_Km", "Hospital_Stay_Days"]
print(clean[cols].corr().round(2))

                          Age   BMI  Fasting_Blood_Sugar  Medications_Count  \
Age                      1.00  0.04                 0.07              -0.06   
BMI                      0.04  1.00                -0.03              -0.05   
Fasting_Blood_Sugar      0.07 -0.03                 1.00               0.14   
Medications_Count       -0.06 -0.05                 0.14               1.00   
Distance_To_Hospital_Km  0.14  0.05                 0.07               0.11   
Hospital_Stay_Days       0.40  0.03                 0.15               0.51   

                         Distance_To_Hospital_Km  Hospital_Stay_Days  
Age                                         0.14                0.40  
BMI                                         0.05                0.03  
Fasting_Blood_Sugar                         0.07                0.15  
Medications_Count                           0.11                0.51  
Distance_To_Hospital_Km                     1.00                0.08  
Hospital_Stay_Days  

In [10]:
clean.to_csv("cleaned_health_data.csv", index=False)
print("Saved cleaned_health_data.csv with shape", clean.shape)

# Read it back to confirm it saved correctly
pd.read_csv("cleaned_health_data.csv").head()

Saved cleaned_health_data.csv with shape (250, 16)


,Patient_ID,City,Gender,Admission_Type,Insurance_Type,Age,BMI,Fasting_Blood_Sugar,Medications_Count,Distance_To_Hospital_Km,Admission_Date,Hospital_Stay_Days,Age_Group,BMI_Category,Admission_Year,Admission_Month
0,P1066,Mumbai,Male,Referral,Private,56,19.3,127.0,8,9.7,2026-05-30,9,Middle Aged,Normal,2026,5
1,P1231,Chennai,Female,Referral,Government,49,25.8,133.0,2,11.0,2025-12-31,6,Middle Aged,Overweight,2025,12
2,P1106,Bengaluru,Male,Emergency,Uninsured,50,25.4,111.0,3,4.5,2025-07-21,7,Middle Aged,Overweight,2025,7
3,P1167,Bengaluru,Male,Emergency,Government,69,28.5,123.0,3,35.4,2025-05-02,10,Senior,Overweight,2025,5
4,P1040,Delhi,Male,Planned,Uninsured,66,24.5,78.0,5,6.6,2026-07-04,9,Senior,Normal,2026,7
